In [1]:
import os, sys, shutil, pickle
import numpy as np

# myoconverter
if 'CONDA_PREFIX' not in os.environ:
    os.environ['CONDA_PREFIX'] = sys.prefix
myoconverter_path = "/home/seojin/biomechanics_tools/myoconverter"
sys.path.append(myoconverter_path)
from myoconverter.optimization.utils.UtilsForceOpt import getMuscleForceDiff, fmOptPSO_cust

# mujoco
if shutil.which("nvidia-smi") is not None:
    os.environ["MUJOCO_GL"] = "egl"
import mujoco

In [2]:
osim_model_path = "/mnt/sdb2/DeepProprioception/OpenSim/MOBL_ARMS_41.osim"
converted_model_path = "/mnt/ext1/seojin/temp/MOBL_ARMS_41_rotated/temp_cvt2.xml"
muscle_dir_path = "/mnt/ext1/seojin/temp/MOBL_ARMS_41_rotated/Step3_muscleKinetics"

In [3]:
model = mujoco.MjModel.from_xml_path(converted_model_path)
data = mujoco.MjData(model)

In [4]:
muscle_names = [model.actuator(i).name for i in range(model.nu)]

['l_mt_min', 'l_mt_max', 'F_max', 'F_scale', 'l_m_min', 'l_m_max',
       'v_max', 'fp_max', 'fv_max', 'unused']

In [5]:
f_max_i = 2
f_scale_i = 3
for control_i in range(model.nu):
    model.actuator_gainprm[control_i][f_max_i] = 1
    model.actuator_gainprm[control_i][f_scale_i] = 1

In [8]:
muscle_name = "CORB" #muscle_names[0]
cvt3_model_path = "/mnt/ext1/seojin/temp/MOBL_ARMS_41_bounded/test_cvt3.xml"
save_dir_path = "/mnt/ext1/seojin/temp/MOBL_ARMS_41_bounded/test"
os.makedirs(save_dir_path, exist_ok = True)

muscle_info_path = os.path.join(muscle_dir_path, f"{muscle_name}.pkl")
with open(muscle_info_path, "rb") as f:
    muscle_para = pickle.load(f)

joints_uniq = muscle_para["jit_uniq"]     
mjc_jnt_arr = muscle_para["mjc_jit_list_set"]
act_arr = muscle_para["act_list"]

# calcualte motion range based on fiber_opt and tendon_slack 
fiber_opt = muscle_para['mtu_par_set']['fiber_opt']
tendon_slack = muscle_para['mtu_par_set']['tendon_sla']

fiber_len_range = np.zeros(2)

# use the osim mtu lengths to calcualte the mujoco muscle parameters
osim_mtu_len_arr = muscle_para['mtu_length_osim']

# if the muscle length is constant, then separate them a bit, since mjc cannot handle equal bounds
if osim_mtu_len_arr[0] == osim_mtu_len_arr[-1]:
    osim_mtu_len_arr[0] = osim_mtu_len_arr[0] - 0.005
    osim_mtu_len_arr[1] = osim_mtu_len_arr[1] + 0.005

# calculate relative fiber lengths
fiber_len_range[0] = (osim_mtu_len_arr[0] - tendon_slack)/fiber_opt
fiber_len_range[1] = (osim_mtu_len_arr[-1] - tendon_slack)/fiber_opt

# make sure that the fiber length range is between [0.01, 1.99], which
# is reasonable
if fiber_len_range[0] < 0.01:
    fiber_len_range[0] = 0.01

if fiber_len_range[1] > 1.99:
    fiber_len_range[1] = 1.99

muscle_inst = model.actuator(muscle_name)
fmax = muscle_para['mtu_par_set']['fmax']
osimFP = muscle_para['mtu_force_osim']/fmax

# actual muscle length
muscle_inst.lengthrange[0] = osim_mtu_len_arr[0]
muscle_inst.lengthrange[1] = osim_mtu_len_arr[-1]

# operation range (nomalized by L0)
muscle_inst.gainprm[0:2] = fiber_len_range

# set lmin and lmax equal 0.0 and 2.0
muscle_inst.gainprm[4] = 0.0
muscle_inst.gainprm[5] = 2.0

# biasprm also NEEDS to be updated
muscle_inst.biasprm = muscle_inst.gainprm

err_ind, mjc_mtu_length, cost_org = getMuscleForceDiff(model, muscle_name,\
                                                              joints_uniq, mjc_jnt_arr,\
                                                              act_arr,\
                                                              osimFP)

if err_ind:
    # set the actual muscle lengths based on the mesh points
    if mjc_mtu_length.min() == mjc_mtu_length.max():
        min_mjc_mtu_length = mjc_mtu_length.min() - 0.005
        max_mjc_mtu_length = mjc_mtu_length.max() + 0.005
    else:
        min_mjc_mtu_length = mjc_mtu_length.min()
        max_mjc_mtu_length = mjc_mtu_length.max()

    muscle_inst.lengthrange[0] = min_mjc_mtu_length
    muscle_inst.lengthrange[1] = max_mjc_mtu_length
    
    # set lmin and lmax as 0 and 1.8
    muscle_inst.gainprm[4:6] = [0, 2]
    muscle_inst.biasprm[4:6] = [0, 2]

    # save current mujoco model into cvt3...
    with open(cvt3_model_path, 'w+') as xml_file:
        mujoco.mj_saveLastXML(cvt3_model_path, model)

    # TODO: 
    # set the muscle activation shift from 0 to 1
    # act_list = np.linspace(0, 1, 5)
    
    
    # only optimize when activation = 1

    # generate parameter bounds
    # If the range value equals the muscle extrem operation length, may cause
    # negative muscle forces, therefore the force-length curve range must be
    # larger the operation range. Adding/Substracting a small number (0.1) to 
    # ensure this.

    optParam_lb = [0.01, 1,\
                   0.01, 0.5]  # lower bounds of lmin, lmax, fpmax, fmax
    optParam_ub = [1, 1.99,\
                   5, 2]  # upper bounds of lmin, lmax, fpmax, fmax
    opt_results, mjc_model = fmOptPSO_cust(cvt3_model_path, muscle_name, joints_uniq,\
                                           mjc_jnt_arr, act_arr, osimFP,\
                                           optParam_lb, optParam_ub,\
                                           cost_org)

    muscle_para['opt_results'] = opt_results

# save the data based to the muscle parameter file
with open(os.path.join(save_dir_path, f"{muscle_name}.pkl"), 'wb') as muscle_file_saving:
    pickle.dump(muscle_para, muscle_file_saving)

# extract muscle instance again.
muscle_inst = mjc_model.actuator(muscle_name)

# change fmax back to normalized value
if len(muscle_para['opt_results']['res_opt']) > 0:
    muscle_inst.gainprm[2] = muscle_para['opt_results']['res_opt'][3]*fmax
else:
    muscle_inst.gainprm[2] = fmax
    
# change vmax to 10*L0, and fvmax as 1.4 (opensim default)
muscle_inst.gainprm[6] = 10
muscle_inst.gainprm[8] = 1.4

# copy the parameter from gainprm to biasprm
muscle_inst.biasprm = muscle_inst.gainprm

2026-05-25 11:20:29.819 | INFO     | myoconverter.optimization.utils.UtilsForceOpt:fmOptPSO_cust:189 -         PSO iteration: 0 ; 0 percentage similarities; Best obj: 796.74362
2026-05-25 11:20:29.876 | INFO     | myoconverter.optimization.utils.UtilsForceOpt:fmOptPSO_cust:189 -         PSO iteration: 1 ; 28 percentage similarities; Best obj: 796.3973
2026-05-25 11:20:29.919 | INFO     | myoconverter.optimization.utils.UtilsForceOpt:fmOptPSO_cust:189 -         PSO iteration: 2 ; 34 percentage similarities; Best obj: 795.06966
2026-05-25 11:20:29.958 | INFO     | myoconverter.optimization.utils.UtilsForceOpt:fmOptPSO_cust:189 -         PSO iteration: 3 ; 30 percentage similarities; Best obj: 794.73636
2026-05-25 11:20:29.959 | INFO     | myoconverter.optimization.utils.UtilsForceOpt:fmOptPSO_cust:197 -         Break the optimization, since certain number of similar particles reached


In [9]:
muscle_inst

<_MjModelActuatorViews
  acc0: array([12.22330921])
  actadr: array([4], dtype=int32)
  actlimited: array([0], dtype=uint8)
  actnum: array([1], dtype=int32)
  actrange: array([0., 0.])
  biasprm: array([4.58388531e-01, 1.87657109e+00, 2.15160000e+03, 1.00000000e+00,
       0.00000000e+00, 1.80000000e+00, 1.00000000e+01, 5.00000000e+00,
       1.40000000e+00, 0.00000000e+00])
  biastype: array([2], dtype=int32)
  cranklength: array([0.])
  ctrllimited: array([1], dtype=uint8)
  ctrlrange: array([0., 1.])
  dynprm: array([0.01, 0.04, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ])
  dyntype: array([3], dtype=int32)
  forcelimited: array([0], dtype=uint8)
  forcerange: array([0., 0.])
  gainprm: array([4.58388531e-01, 1.87657109e+00, 2.15160000e+03, 1.00000000e+00,
       0.00000000e+00, 1.80000000e+00, 1.00000000e+01, 5.00000000e+00,
       1.40000000e+00, 0.00000000e+00])
  gaintype: array([2], dtype=int32)
  gear: array([1., 0., 0., 0., 0., 0.])
  group: array([0], dtype=int32)
  id: